# 06 — Hybrid Model (RF + SVM Ensemble)
Combine Random Forest and SVM using Stacking and Voting for improved accuracy.

**This is the novelty of the project.**

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

X_train, X_test, y_train, y_test = joblib.load('../../models/splits.pkl')
le = joblib.load('../../models/label_encoder.pkl')
class_names = le.classes_
os.makedirs('../../outputs/plots', exist_ok=True)
print("Data loaded. Classes:", class_names)

Data loaded. Classes: ['Healthy' 'High Stress' 'Moderate Stress']


## Method 1 — Voting Classifier
Both RF and SVM vote — class with highest combined probability wins.

In [ ]:
rf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)

voting = VotingClassifier(
    estimators=[('rf', rf), ('svm', svm)],
    voting='soft'   # uses probability scores
)
voting.fit(X_train, y_train)

y_pred_voting = voting.predict(X_test)

acc_voting = accuracy_score(y_test, y_pred_voting)


train_acc_voting = accuracy_score(y_train, voting.predict(X_train))

gap_voting = train_acc_voting - acc_voting

print(f"Voting Train Accuracy: {train_acc_voting * 100:.2f}%")
print(f"Voting Test Accuracy:  {acc_voting * 100:.2f}%")
print(f"Generalization Gap: {gap_voting * 100:.2f}%")

print("Status:", "Potential overfitting" if gap_voting > 0.03 else "No major overfitting signal")
print(classification_report(y_test, y_pred_voting, target_names=class_names))

Voting Train Accuracy: 100.00%
Voting Test Accuracy:  95.42%
Generalization Gap: 4.58%
Status: Potential overfitting
                 precision    recall  f1-score   support

        Healthy       0.98      0.92      0.95        60
    High Stress       0.95      1.00      0.98       100
Moderate Stress       0.94      0.93      0.93        80

       accuracy                           0.95       240
      macro avg       0.96      0.95      0.95       240
   weighted avg       0.95      0.95      0.95       240



## Method 2 — Stacking Classifier
RF and SVM predictions are fed into Logistic Regression as a meta-model.

In [ ]:
rf2  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
svm2 = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)

stacking = StackingClassifier(
    estimators=[('rf', rf2), ('svm', svm2)],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    passthrough=False
)
stacking.fit(X_train, y_train)

y_pred_stack = stacking.predict(X_test)

acc_stack = accuracy_score(y_test, y_pred_stack)


train_acc_stack = accuracy_score(y_train, stacking.predict(X_train))


gap_stack = train_acc_stack - acc_stack

print(f"Stacking Train Accuracy: {train_acc_stack * 100:.2f}%")
print(f"Stacking Test Accuracy:  {acc_stack * 100:.2f}%")
print(f"Generalization Gap: {gap_stack * 100:.2f}%")
print("Status:", "Potential overfitting" if gap_stack > 0.03 else "No major overfitting signal")
print(classification_report(y_test, y_pred_stack, target_names=class_names))

Stacking Train Accuracy: 100.00%
Stacking Test Accuracy:  100.00%
Generalization Gap: 0.00%
Status: No major overfitting signal
                 precision    recall  f1-score   support

        Healthy       1.00      1.00      1.00        60
    High Stress       1.00      1.00      1.00       100
Moderate Stress       1.00      1.00      1.00        80

       accuracy                           1.00       240
      macro avg       1.00      1.00      1.00       240
   weighted avg       1.00      1.00      1.00       240



## Method 3 — Weighted Probability Averaging
Give RF 60% weight and SVM 40% weight based on their individual performance.

In [5]:
# Create the two base models.
# RF = Random Forest, SVM = Support Vector Machine.
rf3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
svm3 = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)

# Train both models on the full training set.
# X_train contains all training samples.
# y_train contains the true label for each training sample.
rf3.fit(X_train, y_train)
svm3.fit(X_train, y_train)

# Get class probabilities for every sample in the test set.
# Shape is usually: (number_of_test_samples, number_of_classes)
rf_proba = rf3.predict_proba(X_test)
svm_proba = svm3.predict_proba(X_test)

# Example meaning of one row:
# rf_proba[0] = [0.1, 0.7, 0.2]
# healthy   #early_blight   #late_blight
# If class order is ['healthy', 'early_blight', 'late_blight'],
# then for test sample 1 the RF model says:
# healthy -> 0.1
# early_blight -> 0.7
# late_blight -> 0.2

# Weighted soft voting:
# give 60% importance to Random Forest
# give 40% importance to SVM
# This is done element-wise for every class probability.
combined = (0.6 * rf_proba) + (0.4 * svm_proba)

# Example for one sample:
# rf_proba[0]  = [0.1, 0.7, 0.2]
# svm_proba[0] = [0.2, 0.6, 0.2]
# combined[0]  = [0.14, 0.66, 0.20]

# Convert combined probabilities into final class prediction.
# np.argmax(..., axis=1) picks the index of the highest probability in each row.
# Example: [0.14, 0.66, 0.20] -> index 1
y_pred_weighted = np.argmax(combined, axis=1) #take column values

# Test accuracy of the weighted model.
acc_weighted = accuracy_score(y_test, y_pred_weighted)

# Do the same weighted prediction on training data
# so we can compare train accuracy vs test accuracy.
rf_train_proba = rf3.predict_proba(X_train)
svm_train_proba = svm3.predict_proba(X_train)

combined_train = (0.6 * rf_train_proba) + (0.4 * svm_train_proba)
y_pred_weighted_train = np.argmax(combined_train, axis=1)

# Training accuracy
train_acc_weighted = accuracy_score(y_train, y_pred_weighted_train)

# Difference between train and test accuracy.
# A large gap can indicate overfitting.
gap_weighted = train_acc_weighted - acc_weighted #testing

print(f"Weighted Train Accuracy: {train_acc_weighted * 100:.2f}%")
print(f"Weighted Test Accuracy:  {acc_weighted * 100:.2f}%")
print(f"Generalization Gap: {gap_weighted * 100:.2f}%")

print("Status:", "Potential overfitting" if gap_weighted > 0.03 else "No major overfitting signal")

# Detailed precision / recall / f1-score for each class
print(classification_report(y_test, y_pred_weighted, target_names=class_names))

NameError: name 'RandomForestClassifier' is not defined

## Confusion Matrix — Best Hybrid Model

In [1]:
# Pick best hybrid
results = {
    'Voting':   (acc_voting,    y_pred_voting),
    'Stacking': (acc_stack,     y_pred_stack),
    'Weighted': (acc_weighted,  y_pred_weighted),
}


best_name = max(results, key=lambda k: results[k][0])
best_acc, best_pred = results[best_name]
print(f"Best Hybrid Method: {best_name} ({best_acc*100:.2f}%)")

plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, best_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=class_names, yticklabels=class_names, linewidths=0.5)
plt.title(f'Hybrid ({best_name}) — Confusion Matrix ({best_acc*100:.2f}%)',
          fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../../outputs/plots/confusion_matrix_hybrid.png', dpi=150)
plt.show()

NameError: name 'acc_voting' is not defined

## Cross-Validation — Best Hybrid

In [20]:
best_model = stacking if best_name == 'Stacking' else voting
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"Hybrid CV Scores: {cv_scores.round(4)}")
print(f"Mean CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

Hybrid CV Scores: [0.9896 0.9896 0.9948 1.     1.    ]
Mean CV Accuracy: 99.48% ± 0.47%


In [21]:
# Save best hybrid model
joblib.dump(stacking, '../../models/hybrid_stacking_model.pkl')
joblib.dump(voting,   '../../models/hybrid_voting_model.pkl')
print("Saved: hybrid_stacking_model.pkl, hybrid_voting_model.pkl")

Saved: hybrid_stacking_model.pkl, hybrid_voting_model.pkl
